# CIDA-Swarm — шаг 1: рой без связи (базовая линия)

Рой дронов учится искать людей в тумане через MAPPO. Здесь две базовые линии, с которыми потом будет
сравниваться CIDA-связь:

* **nocomm** — каждый дрон сам за себя (только свои глаза и своя память);
* **stigmergy** — дроны видят общую карту исследованных клеток (как феромон у муравьёв).

Архитектура и план экспериментов: `docs/CIDA_SWARM_ARCHITECTURE.md`.
Runtime → Change runtime type → **A100** (подойдёт и L4). Результаты сохраняются на Google Drive.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import os
if not os.path.exists("/content/CIDA-plugin"):
    !git clone https://github.com/Kairatzh/CIDA-plugin.git /content/CIDA-plugin
%cd /content/CIDA-plugin
!git pull --ff-only
!git log --oneline -1

In [ ]:
# runs/ на Google Drive: если Colab отключится, готовые прогоны не пропадут
from google.colab import drive
drive.mount("/content/drive")
RUNS = "/content/drive/MyDrive/cida_swarm_runs"
os.makedirs(RUNS, exist_ok=True)

In [ ]:
PRESET = "full"   # "quick" — проверка за ~10 минут, "full" — ночной прогон

# усложнённая среда: маленький обзор и медленные дроны, иначе рой без связи решает всё сам
ENV = "--view-radius 0.08 --max-speed 0.015 --max-steps 150 --n-targets 8 --n-drones 6"
if PRESET == "quick":
    COMMON = f"{ENV} --n-envs 1024 --iters 200 --eval-every 100"
    SEEDS = [0]
else:
    COMMON = f"{ENV} --n-envs 2048 --iters 3000 --eval-every 250"
    SEEDS = [0, 1, 2]

EXPERIMENTS = {
    "nocomm": "",
    "stigmergy": "--share-coverage",
}

In [ ]:
import sys
from IPython import get_ipython
for seed in SEEDS:
    for name, flags in EXPERIMENTS.items():
        run = f"{name}_s{seed}"
        out = f"{RUNS}/{run}"
        if os.path.exists(f"{out}/result.json"):
            print("skip", run); continue
        cmd = f"{sys.executable} -u -m cida_swarm.train {flags} {COMMON} --seed {seed} --name {run} --out {out}"
        print(">>>", cmd, flush=True)
        get_ipython().system(cmd)
        if not os.path.exists(f"{out}/result.json"):
            raise RuntimeError(f"{run} не завершился — смотрите лог выше")

In [ ]:
import glob, json, re
from collections import defaultdict
import numpy as np

groups = defaultdict(list)
for path in sorted(glob.glob(f"{RUNS}/*/result.json")):
    r = json.load(open(path))
    groups[re.sub(r"_s\d+$", "", r["name"])].append(r)

def fmt(xs):
    return f"{np.mean(xs):.3f} ± {np.std(xs, ddof=1):.3f}" if len(xs) > 1 else f"{xs[0]:.3f}"

print("| Run | Seeds | Found | All found | Steps | Coverage | Random found |")
print("|---|---|---|---|---|---|---|")
for name, runs in sorted(groups.items()):
    f = [r["final"] for r in runs]
    print(f"| {name} | {len(runs)} | {fmt([x['found_frac'] for x in f])} | {fmt([x['all_found'] for x in f])} "
          f"| {fmt([x['length'] for x in f])} | {fmt([x['coverage'] for x in f])} "
          f"| {fmt([r['random']['found_frac'] for r in runs])} |")

In [ ]:
import matplotlib.pyplot as plt
for name, runs in sorted(groups.items()):
    for r in runs:
        h = r["history"]
        plt.plot([x["agent_steps"] / 1e6 for x in h], [x["found_frac"] for x in h],
                 label=r["name"], color="C0" if name == "nocomm" else "C1", alpha=0.8)
plt.xlabel("млн шагов дронов"); plt.ylabel("доля найденных людей"); plt.legend(); plt.grid(alpha=0.3)
plt.show()

## Как читать

* **Found** — доля найденных людей за эпизод, **All found** — доля миров, где нашли всех, **Steps** — длина эпизода
  (меньше = быстрее нашли всех).
* Базовая линия хороша, если она заметно лучше случайной, но **не решает задачу полностью**: должно оставаться место,
  где связь CIDA может помочь. Если `nocomm` уже находит почти всех — среду нужно усложнить.
* Пришлите таблицу и строку `agent-steps/s` из лога: по ним спланируем шаг 2 (CIDA-связь).